# Compact `sales.orders`

Every write to an Iceberg table adds a new (often small) data file. Query
engines have to open every one of them, so a table written to often — a few
rows at a time, the way this demo has been — accumulates many small files and
gets slower to scan. **Compaction** rewrites a table's small files into fewer,
larger ones without changing any data, via Iceberg's `rewrite_data_files`
stored procedure (Spark + Iceberg's table-maintenance mechanism, not something
specific to this demo stack).

Run this notebook **locally** (not in Docker) with `docker compose up -d`
already running. Setup: `cd spark && uv sync && uv run jupyter notebook`.
Requires Java 17+ and the one-time DNS entry from `docs/CORS_issues.md`:
```bash
echo "127.0.0.1 minio.localhost" | sudo tee -a /etc/hosts
```

Compaction rewrites files and commits a new snapshot, so it needs `modify`
permission on the table — the same permission a normal write needs (see
`reconciler/grants.yaml`). It reads `SPARK_USERNAME`/`SPARK_PASSWORD` from
`spark/.env` if present (copy `.env.example` to set one up), else prompts.

In [ ]:
username = "alice"

In [2]:
username

NameError: name 'username' is not defined

In [ ]:
# Shared with query_orders.py so there is one implementation.
from query_orders import get_credentials, get_spark_session

username, password = get_credentials()
print(f"Using {username}.")

In [ ]:
spark = get_spark_session(username, password, "lakehouse-compact-orders")

## 1. Before: current data files

Iceberg exposes table metadata as queryable tables under `<table>.files`,
`<table>.snapshots`, etc. — ordinary SQL, no special tooling needed.

In [ ]:
before = spark.sql("SELECT file_path, file_size_in_bytes, record_count FROM sales.orders.files")
before.show(truncate=False)
print(f"{before.count()} data file(s) before compaction.")

## 2. Compact

`rewrite_data_files` is an Iceberg Spark-extensions stored procedure, called
with SQL `CALL`. `min-input-files` lowers the default threshold (5) so this
small demo table actually has something to compact; leave it at the default
in a real table sized to matter.

In [ ]:
result = spark.sql(
    """
    CALL lakekeeper.system.rewrite_data_files(
        table => 'sales.orders',
        options => map('min-input-files', '2')
    )
    """
)
result.show(truncate=False)

## 3. After: fewer, larger files

Same row count, same data — verify both. Fewer files, and a new `replace`
snapshot recording the rewrite (the pre-compaction snapshots are still there
for time travel until they age out / are expired).

In [ ]:
after = spark.sql("SELECT file_path, file_size_in_bytes, record_count FROM sales.orders.files")
after.show(truncate=False)
print(f"{after.count()} data file(s) after compaction.")

In [ ]:
spark.sql("SELECT * FROM sales.orders ORDER BY order_id").show()

In [ ]:
spark.sql(
    "SELECT committed_at, operation, summary['total-data-files'] AS total_data_files "
    "FROM sales.orders.snapshots ORDER BY committed_at"
).show(truncate=False)